# COLLABORATIVE FILTERING WITH IMPLICIT FEEDBACK

In [2]:
import pandas as pd

In [ ]:
#Step 1
transactions = pd.read_csv("data/transactions_6_months.csv")
transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

In [ ]:
#Step 2
transactions = transactions.sort_values("t_dat")

# Get maximum date
max_date = transactions["t_dat"].max()

# Last month for validation
split_date = max_date - pd.DateOffset(months=1)
train_df = transactions[
    transactions["t_dat"] < split_date
]

valid_df = transactions[
    transactions["t_dat"] >= split_date
]

In [ ]:
#Step 3
interaction_df = (
    train_df
    .groupby(["customer_id", "article_id"])
    .size()
    .reset_index(name="strength")
)

In [ ]:
#Step 4
user_ids = interaction_df["customer_id"].unique()
item_ids = interaction_df["article_id"].unique()

user_to_idx = {
    x: i for i, x in enumerate(user_ids)
}

item_to_idx = {
    x: i for i, x in enumerate(item_ids)
}

In [8]:
interaction_df["user_idx"] = (
    interaction_df["customer_id"]
    .map(user_to_idx)
)

interaction_df["item_idx"] = (
    interaction_df["article_id"]
    .map(item_to_idx)
)

In [9]:
#Step 5
from scipy.sparse import csr_matrix

user_item_matrix = csr_matrix(
    (
        interaction_df["strength"],
        (
            interaction_df["user_idx"],
            interaction_df["item_idx"]
        )
    )
)

In [10]:
#Step 6
from implicit.als import AlternatingLeastSquares

alpha = 40

model = AlternatingLeastSquares(
    factors=128,
    regularization=0.01,
    iterations=20,
    random_state=42
)

model.fit(user_item_matrix * alpha)

c:\Users\trinhhung\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\trinhhung\AppData\Local\Programs\Python\Python312\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 20/20 [01:57<00:00,  5.86s/it]


In [11]:
#Step 7
customer_id = train_df["customer_id"].iloc[0]

user_idx = user_to_idx[customer_id]

ids, scores = model.recommend(
    userid=user_idx,
    user_items=user_item_matrix[user_idx],
    N=12
)

In [12]:
idx_to_item = {
    idx: item
    for item, idx in item_to_idx.items()
}

recommendations = [
    idx_to_item[i]
    for i in ids
]

In [13]:
# Choose a customer
customer_id = train_df["customer_id"].iloc[0]

# Convert customer_id → user index
user_idx = user_to_idx[customer_id]

# Generate recommendations
ids, scores = model.recommend(
    userid=user_idx,
    user_items=user_item_matrix[user_idx],
    N=12,
    filter_already_liked_items=False
)

# Convert item indices back to article IDs
recommendations = [
    idx_to_item[item_idx]
    for item_idx in ids
]

# Print results
print("=" * 50)
print("CUSTOMER ID:")
print(customer_id)

print("\nTOP 12 RECOMMENDATIONS")
print("-" * 50)

for rank, (article_id, score) in enumerate(zip(recommendations, scores), start=1):
    print(f"{rank:02d}. Article ID: {article_id} | Score: {score:.4f}")

CUSTOMER ID:
000d15e5f8031a32013ab1e4557854692557a2aa7c79265f027265503a3efac5

TOP 12 RECOMMENDATIONS
--------------------------------------------------
01. Article ID: 806388003 | Score: 0.6757
02. Article ID: 803757013 | Score: 0.6522
03. Article ID: 806388005 | Score: 0.6204
04. Article ID: 795440003 | Score: 0.6188
05. Article ID: 806388002 | Score: 0.6066
06. Article ID: 796210012 | Score: 0.5933
07. Article ID: 806388011 | Score: 0.5807
08. Article ID: 796210001 | Score: 0.5563
09. Article ID: 806388009 | Score: 0.5550
10. Article ID: 806388001 | Score: 0.5195
11. Article ID: 803757011 | Score: 0.4955
12. Article ID: 786304005 | Score: 0.4752


### CHECK PRECISION ,RECALL AND NDCG

In [14]:
# Actual purchased items in validation set

ground_truth = (
    valid_df
    .groupby("customer_id")["article_id"]
    .apply(list)
    .to_dict()
)

In [15]:
K = 12

predictions = {}

for customer_id in ground_truth.keys():

    # Skip unseen users
    if customer_id not in user_to_idx:
        continue

    user_idx = user_to_idx[customer_id]

    ids, scores = model.recommend(
        userid=user_idx,
        user_items=user_item_matrix[user_idx],
        N=K,
        filter_already_liked_items=False
    )

    recs = [
        idx_to_item[item_idx]
        for item_idx in ids
    ]

    predictions[customer_id] = recs

In [16]:
def precision_at_k(actual, predicted, k=12):

    actual_set = set(actual)
    pred_set = set(predicted[:k])

    return len(actual_set & pred_set) / k

In [17]:
def recall_at_k(actual, predicted, k=12):

    actual_set = set(actual)
    pred_set = set(predicted[:k])

    if len(actual_set) == 0:
        return 0

    return len(actual_set & pred_set) / len(actual_set)

In [18]:
import numpy as np

def ndcg_at_k(actual, predicted, k=12):

    actual_set = set(actual)
    predicted = predicted[:k]

    dcg = 0.0

    for i, item in enumerate(predicted):

        if item in actual_set:
            dcg += 1 / np.log2(i + 2)

    # Ideal DCG
    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(min(len(actual_set), k))
    )

    if idcg == 0:
        return 0

    return dcg / idcg

In [19]:
precision_scores = []
recall_scores = []
ndcg_scores = []

for customer_id in predictions:

    actual = ground_truth[customer_id]
    predicted = predictions[customer_id]

    precision_scores.append(
        precision_at_k(actual, predicted, K)
    )

    recall_scores.append(
        recall_at_k(actual, predicted, K)
    )

    ndcg_scores.append(
        ndcg_at_k(actual, predicted, K)
    )

In [20]:
print("=" * 50)
print("ALS EVALUATION")
print("=" * 50)

print(f"Precision@{K}: {np.mean(precision_scores):.4f}")
print(f"Recall@{K}:    {np.mean(recall_scores):.4f}")
print(f"NDCG@{K}:      {np.mean(ndcg_scores):.4f}") 

ALS EVALUATION
Precision@12: 0.0055
Recall@12:    0.0201
NDCG@12:      0.0143
